## Deliverable 2. Create a Customer Travel Destinations Map.

In [1]:
# Dependencies and Setup
import pandas as pd
import requests
import gmaps

In [2]:
# Import API key
from config import g_key

# Configure gmaps API key
gmaps.configure(api_key=g_key)

In [3]:
# 1. Import the WeatherPy_database.csv file. 
city_data_df = pd.read_csv("../Weather_Database/WeatherPy_database.csv")
city_data_df.head()

,City_ID,City,Country,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Current Description
0,0,East London,ZA,-33.0153,27.9116,58.03,50,0,7.45,clear sky
1,1,Ponta Do Sol,PT,32.6667,-17.1000,67.01,66,34,7.63,scattered clouds
2,2,Yar-Sale,RU,66.8333,70.8333,53.51,48,100,27.40,overcast clouds
3,3,Garowe,SO,8.4054,48.4845,76.91,57,41,31.88,scattered clouds
4,4,Roccastrada,IT,43.0099,11.1665,74.01,90,16,1.57,few clouds


In [10]:
# 2. Prompt the user to enter minimum and maximum temperature criteria 

min_temp = float(input("What is the minimum temperature you would like for your trip? "))

max_temp = float(input("What is the maximum temperature you would like for your trip? "))


What is the minimum temperature you would like for your trip? 75
What is the maximum temperature you would like for your trip? 90


In [11]:
# 3. Filter the city_data_df DataFrame using the input statements to create a new DataFrame using the loc method.

preferred_cities_df = city_data_df.loc[(city_data_df["Max Temp"] <= max_temp) & \
                                       (city_data_df["Max Temp"] >= min_temp)]

preferred_cities_df.head(10)


,City_ID,City,Country,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Current Description
3,3,Garowe,SO,8.4054,48.4845,76.91,57,41,31.88,scattered clouds
6,6,Pasni,PK,25.2631,63.4710,82.08,90,26,8.79,scattered clouds
8,8,Victoria,HK,22.2855,114.1577,86.25,83,43,5.99,scattered clouds
9,9,Hithadhoo,MV,-0.6000,73.0833,82.94,73,100,9.93,overcast clouds
10,10,Acarau,BR,-2.8856,-40.1200,78.28,85,60,11.12,broken clouds
13,13,Salalah,OM,17.0151,54.0924,82.49,94,100,2.30,overcast clouds
16,16,Piney Green,US,34.7160,-77.3202,80.55,96,100,3.44,overcast clouds
22,22,Ajdabiya,LY,30.7554,20.2263,83.30,35,0,9.19,clear sky
23,23,Sanchi,IN,23.4833,77.7333,83.77,70,100,6.53,overcast clouds
27,27,Kutum,SD,14.2000,24.6667,81.43,38,81,9.35,broken clouds


In [12]:
# 4a. Determine if there are any empty rows.
preferred_cities_df.count()

City_ID                224
City                   224
Country                224
Lat                    224
Lng                    224
Max Temp               224
Humidity               224
Cloudiness             224
Wind Speed             224
Current Description    224
dtype: int64

In [13]:
# 4b. Drop any empty rows and create a new DataFrame that doesn’t have empty rows.
preferred_cities_clean_df = preferred_cities_df.dropna()
preferred_cities_clean_df.count()

City_ID                224
City                   224
Country                224
Lat                    224
Lng                    224
Max Temp               224
Humidity               224
Cloudiness             224
Wind Speed             224
Current Description    224
dtype: int64

In [14]:
# 5a. Create DataFrame called hotel_df to store hotel names along with city, country, max temp, and coordinates.
hotel_df = preferred_cities_clean_df[["City", "Country", "Max Temp", "Current Description", "Lat", "Lng"]].copy()

# 5b. Create a new column "Hotel Name"
hotel_df["Hotel Name"] = ""
hotel_df.tail(10)

,City,Country,Max Temp,Current Description,Lat,Lng,Hotel Name
675,Sao Joao Do Piaui,BR,78.42,clear sky,-8.3581,-42.2467,
679,Alibag,IN,80.67,light rain,18.6411,72.8792,
681,Safaga,EG,87.82,clear sky,26.7292,33.9365,
693,Khilchipur,IN,81.75,overcast clouds,24.0333,76.5667,
696,Rock Sound,BS,80.28,overcast clouds,24.9000,-76.2000,
701,Damghan,IR,85.71,clear sky,36.1683,54.3480,
702,Nieuw Amsterdam,SR,80.76,scattered clouds,5.8833,-55.0833,
707,Chardara,KZ,82.98,clear sky,41.2547,67.9692,
709,Manadhoo,MV,82.78,few clouds,5.7667,73.3833,
710,Pansemal,IN,78.49,overcast clouds,21.6500,74.7000,


In [15]:
# 6a. Set parameters to search for hotels with 5000 meters.
params = {
    "radius": 5000,
    "type": "lodging",
    "key": g_key
}

# 6b. Iterate through the hotel DataFrame.

for index, row in hotel_df.iterrows():
    
    # 6c. Get latitude and longitude from DataFrame

    lat = row["Lat"]
    lng = row["Lng"]
    
    params["location"] = f"{lat},{lng}"
    
    # 6d. Set up the base URL for the Google Directions API to get JSON data.
    
    base_url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"

    # 6e. Make request and retrieve the JSON data from the search. 
   
    hotels = requests.get(base_url, params=params).json()
    
    # 6f. Get the first hotel from the results and store the name, if a hotel isn't found skip the city.
    
    try:
        hotel_df.loc[index, "Hotel Name"] = hotels["results"][0]["name"]
    except (IndexError):
        print("Hotel not found... skipping.")

Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.


In [16]:
# 7. Drop the rows where there is no Hotel Name.

hotel_df["Hotel Name"].astype(bool)

hotel_clean_df = hotel_df[hotel_df["Hotel Name"].astype(bool)]

hotel_clean_df.count()

City                   210
Country                210
Max Temp               210
Current Description    210
Lat                    210
Lng                    210
Hotel Name             210
dtype: int64

In [17]:
# 8a. Create the output File (CSV)

output_data_file = "../Vacation_Search/WeatherPy_vacation.csv"

# 8b. Export the City_Data into a csv
hotel_clean_df.to_csv(output_data_file, index_label="City_ID")

In [18]:
# 9. Using the template add city name, the country code, the weather description and maximum temperature for the city.
info_box_template = """

<dl>

<dt>Hotel Name</dt><dd>{Hotel Name}</dd>

<dt>City</dt><dd>{City}</dd>

<dt>Country</dt><dd>{Country}</dd>

<dt>Current Weather</dt><dd>{Current Description} and {Max Temp} °F</dd>


</dl>

"""

# 10a. Get the data from each row and add it to the formatting template and store the data in a list.
hotel_info = [info_box_template.format(**row) for index, row in hotel_clean_df.iterrows()]

# 10b. Get the latitude and longitude from each row and store in a new DataFrame.
locations = hotel_clean_df[["Lat", "Lng"]]

In [19]:
# 11a. Add a marker layer for each city to the map. 

# Add a marker layer map for the vacation spots and a pop-up marker for each city.

locations = hotel_clean_df[["Lat", "Lng"]]

max_temp = hotel_clean_df["Max Temp"]

fig = gmaps.figure(center=(30.0, 31.0), zoom_level=1.5)

marker_layer = gmaps.marker_layer(locations, info_box_content=hotel_info)

fig.add_layer(marker_layer)


# 11b. Display the figure

fig


Figure(layout=FigureLayout(height='420px'))